# Part 3 · Notebook 04 — Typed messages and asyncio

**Sessions:** S6 (Type hints, dataclasses, enums & Pydantic) · S7 (Concurrency & asyncio) · Clinic W2 · [Lesson plan](../../docs/lessons/PART_03_PYTHON_ENGINEERING.md) · graded labs in [`labs/part03/`](../../labs/part03/)

**You will:**
1. Model data with frozen dataclasses and enums.
2. Validate broker messages with Pydantic, including a rule across fields.
3. Run I/O-bound calls concurrently with `asyncio.gather`.
4. See back-pressure: what a bounded queue does when the consumer is slow.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p3lib.py is in notebooks/part03/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p3lib as p

p.use_course_style()

## 1. Dataclasses and enums

In [ ]:
from dataclasses import dataclass, FrozenInstanceError
from enum import Enum

class Side(str, Enum):
    BUY = "BUY"
    SELL = "SELL"

@dataclass(frozen=True, slots=True)
class Fill:
    symbol: str
    side: Side
    qty: int
    price: Decimal

f = Fill("SPY", Side.BUY, 100, Decimal("512.10"))
print(f, Side("SELL"), f == Fill("SPY", Side.BUY, 100, Decimal("512.10")))
try:
    f.qty = 200
except FrozenInstanceError as e:
    print("frozen:", e)

## 2. Validating messages with Pydantic

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field, ValidationError, model_validator

class OrderRequest(BaseModel):
    symbol: str = Field(min_length=1, max_length=12)
    side: Literal["BUY", "SELL"]
    qty: int = Field(gt=0)
    order_type: Literal["MKT", "LMT"]
    limit_price: Decimal | None = None

    @model_validator(mode="after")
    def price_rules(self):
        # ✍️ LMT needs a limit_price; MKT must NOT have one (raise ValueError with a message)
        ...
        return self

payloads = [{"symbol": "SPY", "side": "BUY", "qty": 100, "order_type": "LMT", "limit_price": "512.10"},
            {"symbol": "SPY", "side": "BUY", "qty": 100, "order_type": "LMT"},
            {"symbol": "SPY", "side": "BUY", "qty": 100, "order_type": "MKT", "limit_price": "500"},
            {"symbol": "SPY", "side": "SELL", "qty": 0, "order_type": "MKT"},
            {"symbol": "QQQ", "side": "SELL", "qty": 5, "order_type": "MKT"}]

def is_valid(d):
    try:
        OrderRequest(**d)
        return True
    except ValidationError:
        return False

valid = p.check("order validation", [is_valid(d) for d in payloads], [True, False, False, False, True])
try:
    OrderRequest(**payloads[1])
except ValidationError as e:
    print(e)

## 3. `asyncio`: waiting for many things at once

Each fake quote request takes 0.2 s of *waiting* (network), not CPU. Sequentially, 5 requests take 1 s.

In [ ]:
import asyncio, time

async def get_quote(symbol: str) -> dict:
    await asyncio.sleep(0.2)                      # the network round trip
    return {"symbol": symbol, "bid": 99.99, "ask": 100.01}

SYMBOLS = ["SPY", "QQQ", "IWM", "TLT", "GLD"]
t0 = time.perf_counter()
seq = [await get_quote(s) for s in SYMBOLS]
print(f"sequential: {time.perf_counter() - t0:.2f} s")

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
t0 = time.perf_counter()
# ✍️ request all SYMBOLS concurrently with asyncio.gather (results in the same order)
quotes = ...
elapsed = time.perf_counter() - t0
print(f"concurrent: {elapsed:.2f} s")
quotes = p.check("asyncio.gather", quotes, seq)
fast = p.check("finished in well under a second", elapsed < 0.6, True)

## 4. Back-pressure with a bounded queue (clinic W2)

A fast feed (1 quote / ms) and a slow consumer (1.5 ms per quote). With an unbounded queue the backlog — and the age of the data you act on — keeps growing. A bounded queue makes the producer wait instead.

In [ ]:
async def run(maxsize: int, n: int = 300):
    q, ages, depth = asyncio.Queue(maxsize=maxsize), [], []

    async def producer():
        for i in range(n):
            await q.put(time.perf_counter())      # waits when a bounded queue is full
            await asyncio.sleep(0.001)
        await q.put(None)

    async def consumer():
        while (ts := await q.get()) is not None:
            await asyncio.sleep(0.0015)           # slow processing
            ages.append((time.perf_counter() - ts) * 1e3)
            depth.append(q.qsize())

    await asyncio.gather(producer(), consumer())
    return np.array(ages), np.array(depth)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for maxsize, label in ((0, "unbounded"), (10, "bounded (10)")):
    ages, depth = await run(maxsize)
    axes[0].plot(ages, label=f"{label}: p99 {np.percentile(ages, 99):.0f} ms")
    axes[1].plot(depth, label=label)
axes[0].set(title="Age of each quote when processed (ms)", xlabel="quote #"); axes[0].legend()
axes[1].set(title="Queue depth", xlabel="quote #"); axes[1].legend()
plt.tight_layout(); plt.show()

## Questions
1. Why is `asyncio` a good fit for talking to brokers and data feeds, but no help for a CPU-heavy backtest?
2. The bounded queue slows the producer. In a live feed you cannot slow the exchange — what would you do instead (drop, conflate to the latest quote, …)?
3. Which Pydantic rule above would you also enforce in the risk engine, and why twice?

**Graded version:** `labs/part03/week08_advanced` (Pydantic `OrderRequest`, async pipeline) and `labs/part03/clinic_w2_async_sim`.